### 라이브러리·경로 설정

In [ ]:
from pathlib import Path
from textwrap import fill
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import Normalizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, make_scorer
from statsmodels.tools.tools import add_constant
from statsmodels.stats.outliers_influence import variance_inflation_factor

data_dir = Path("dataset")
output_dir = Path("outputs/simple_oob")
figure_dir = output_dir / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)
seed = 42
n_trees = 300
class_names = ["Normal", "CRC"]
positive_class = "CRC"
sns.set_theme(style="whitegrid", font="Malgun Gothic")
plt.rcParams["axes.unicode_minus"] = False

### ASV count 로딩

In [ ]:
count_df = pd.read_excel(data_dir / "seqtab_nochim_export.xlsx")
count_df = count_df.rename(columns={count_df.columns[0]: "sample_name"})
print("count 표 크기:", count_df.shape)
display(count_df.iloc[:5, :5])
count_df.info(verbose=False)

### Taxonomy 로딩·ASV ID 부여

In [ ]:
taxonomy_df = pd.read_excel(data_dir / "taxa_species_export.xlsx")
taxonomy_df = taxonomy_df.rename(columns={taxonomy_df.columns[0]: "sequence"})
taxonomy_df.insert(0, "ASV_ID", [f"ASV_{i:06d}" for i in range(1, len(taxonomy_df) + 1)])
print("taxonomy 표 크기:", taxonomy_df.shape)
display(taxonomy_df.head())
display(taxonomy_df.isna().sum().to_frame("결측 수"))

### Metadata 로딩

In [ ]:
metadata_df = pd.read_csv(data_dir / "metadata.csv", sep=",", encoding="utf-8")
print("metadata 표 크기:", metadata_df.shape)
display(metadata_df.head())
display(metadata_df["DiseaseStatus"].value_counts().to_frame("샘플 수"))

### BLAST 결과 로딩

In [ ]:
blast_dir = data_dir / "blast_merged_taxid"
blast_df = pd.read_csv(blast_dir / "blast_hits_merged_taxid.csv",
                       encoding="utf-8", dtype={"subject_taxid": str})
print("BLAST 표 크기:", blast_df.shape)
display(blast_df.head())

### Accession 대응표 로딩

In [ ]:
accession_df = pd.read_csv(blast_dir / "accession_taxid_mapping.csv",
                          encoding="utf-8", dtype={"taxid": str, "ncbi_uid": str})
print("accession 대응표 크기:", accession_df.shape)
display(accession_df.head())

### 샘플·DNA 서열 연결 키 확인

In [ ]:
raw_count_df = count_df.set_index("sample_name")
key_check_df = pd.Series({
    "count 샘플 중복": count_df["sample_name"].duplicated().sum(),
    "metadata 샘플 중복": metadata_df["host_disease"].duplicated().sum(),
    "taxonomy 서열 중복": taxonomy_df["sequence"].duplicated().sum(),
    "count 결측": raw_count_df.isna().sum().sum(),
    "metadata 결측": metadata_df.isna().sum().sum(),
    "count 서열 미매칭": (~raw_count_df.columns.isin(taxonomy_df["sequence"])).sum()
})
display(key_check_df.to_frame("개수"))
display(taxonomy_df[["ASV_ID", "sequence"]].head(3))

### BLAST·TaxID 대응 확인

In [ ]:
reference_df = blast_df[["subject_accession", "subject_taxid"]].drop_duplicates()
reference_df = pd.merge(reference_df, accession_df, left_on="subject_accession",
                         right_on="accession", how="left")
print("accession 미매칭:", reference_df["taxid"].isna().sum())
print("TaxID 불일치:", (reference_df["subject_taxid"] != reference_df["taxid"]).sum())
print("hit 행 수:", len(blast_df), "/ hit가 있는 ASV:", blast_df["ASV_ID"].nunique())

### ASV별 BLAST 최고 점수 행 선택

In [ ]:
# ASV별 최고 점수의 첫 번째 행을 선택합니다.
best_hits_df = blast_df.loc[blast_df.groupby("ASV_ID")["bit_score"].idxmax()].copy()
best_hits_df = pd.merge(best_hits_df, accession_df[["accession", "record_title"]],
                        left_on="subject_accession", right_on="accession", how="left")
sequence_length = taxonomy_df.set_index("ASV_ID")["sequence"].str.len()
best_hits_df["hsp_coverage"] = (
    (best_hits_df["query_end"] - best_hits_df["query_start"]).abs() + 1
) / best_hits_df["ASV_ID"].map(sequence_length) * 100
display(best_hits_df[["ASV_ID", "bit_score", "subject_accession", "subject_scientific_name"]].head())

### ASV 서열·BLAST 참조 학명 대응표

In [ ]:
asv_annotation_df = pd.merge(taxonomy_df, best_hits_df, on="ASV_ID", how="inner")
sequence_to_name = asv_annotation_df.set_index("sequence")["subject_scientific_name"]
print("전체 ASV:", raw_count_df.shape[1], "/ BLAST 매칭:", len(sequence_to_name))
print("미매칭 ASV:", raw_count_df.shape[1] - len(sequence_to_name))
display(asv_annotation_df[["ASV_ID", "sequence", "Species", "subject_scientific_name"]].head())

### DNA 서열 열의 학명 변경·read 합산

In [ ]:
# DNA 서열 열을 BLAST 참조 학명으로 바꾸고 같은 학명의 read를 합칩니다.
matched_count_df = raw_count_df[sequence_to_name.index].rename(columns=sequence_to_name.to_dict())
scientific_count_df = matched_count_df.T.groupby(level=0, sort=True).sum().T
scientific_count_df.columns.name = None
scientific_name_columns = scientific_count_df.columns.tolist()
# 미매칭 read는 정규화 분모에만 포함하고 모델 특성에서는 제외합니다.
scientific_count_df["unmatched_reads"] = raw_count_df.loc[
    :, ~raw_count_df.columns.isin(sequence_to_name.index)].sum(axis=1)
normalization_columns = scientific_name_columns + ["unmatched_reads"]
feature_annotation_df = asv_annotation_df.groupby("subject_scientific_name").size().to_frame("n_asvs")
feature_annotation_df.index.name = "scientific_name"
print("학명 합산 전·후:", matched_count_df.shape[1], len(scientific_name_columns))
print("전체 read 합 보존 차이:", (scientific_count_df.sum(axis=1) - raw_count_df.sum(axis=1)).abs().max())
display(scientific_count_df.iloc[:5, :4])
display(feature_annotation_df.head())

### 학명별 Count·Metadata 통합

In [ ]:
analysis_df = pd.merge(scientific_count_df.reset_index(), metadata_df,
                       left_on="sample_name", right_on="host_disease", how="left")
print("통합 전·후 샘플 수:", len(count_df), len(analysis_df))
print("정답 결측 수:", analysis_df["DiseaseStatus"].isna().sum())
display(analysis_df[["sample_name", *scientific_name_columns[:2], "DiseaseStatus", "Age"]].head())

### 정상·CRC 정답 구성

In [ ]:
target_mapping = {
    "Healthy": "Normal",
    "Adenomatous Polyps": "Normal",
    "Colorectal cancer": "CRC"
}
analysis_df["DiseaseGroup"] = analysis_df["DiseaseStatus"].map(target_mapping)
display(analysis_df[["DiseaseStatus", "DiseaseGroup"]].drop_duplicates())
display(analysis_df["DiseaseGroup"].value_counts().to_frame("샘플 수"))

### 통합 표·서열 대응표 저장

In [ ]:
display(analysis_df[["sample_name", *scientific_name_columns[:3], "DiseaseGroup"]].head(3))
analysis_df.to_csv(output_dir / "analysis_merged.csv", index=False, encoding="utf-8")
scientific_count_df.to_csv(output_dir / "scientific_name_count.csv", encoding="utf-8")
feature_annotation_df.to_csv(output_dir / "scientific_name_features.csv", encoding="utf-8")
asv_annotation_df.to_csv(output_dir / "asv_annotation.csv", index=False, encoding="utf-8")
asv_annotation_df[["ASV_ID", "sequence", "subject_scientific_name"]].to_csv(
    output_dir / "sequence_scientific_name_mapping.csv", index=False, encoding="utf-8")
taxonomy_df[["ASV_ID", "sequence"]].to_csv(output_dir / "asv_id_taxonomy_mapping.csv", index=False, encoding="utf-8")
best_hits_df.to_csv(output_dir / "blast_best_hits.csv", index=False, encoding="utf-8")

### Count 품질·기본 통계

In [ ]:
total_reads = raw_count_df.sum(axis=1)
feature_stats_df = pd.DataFrame({
    "detected_samples": (scientific_count_df[scientific_name_columns] > 0).sum(),
    "max_count": scientific_count_df[scientific_name_columns].max()
})
print("원본 count 결측:", raw_count_df.isna().sum().sum())
print("음수 count:", (raw_count_df < 0).sum().sum())
print("비정수 count:", (raw_count_df % 1 != 0).sum().sum())
print("학명별 count의 0 비율:", round((scientific_count_df[scientific_name_columns] == 0).to_numpy().mean(), 4))
display(total_reads.describe().to_frame("샘플별 총 reads"))
display(analysis_df[["Sex", "Age"]].describe(include="all"))

### 데이터 분포 시각화

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
analysis_df["DiseaseGroup"].value_counts().plot.bar(ax=axes[0, 0], rot=15, title="그룹별 샘플 수")
sns.histplot(total_reads, bins=15, ax=axes[0, 1])
axes[0, 1].set(title="샘플별 총 read 수", xlabel="총 reads")
sns.histplot(feature_stats_df["detected_samples"], bins=20, ax=axes[1, 0])
axes[1, 0].set(title="학명별 검출 샘플 수", xlabel="count가 0보다 큰 샘플 수")
sns.histplot(np.log10(1 + scientific_count_df[scientific_name_columns].to_numpy().ravel()), bins=30, ax=axes[1, 1])
axes[1, 1].set(title="학명별 합산 count: 0 포함", xlabel="log10(1 + count)", yscale="log")
fig.tight_layout()
fig.savefig(figure_dir / "01_data_overview.png", dpi=150)
plt.show()

### Train·Test 분리

In [ ]:
X_count = analysis_df.set_index("sample_name")[normalization_columns]
y = analysis_df.set_index("sample_name")["DiseaseGroup"]

# 원본 질병 그룹을 기준으로 층화합니다.
test_size = 0.3
all_y = analysis_df.set_index("sample_name")["DiseaseStatus"]
train_samples, test_samples = train_test_split(
    all_y.index, test_size=test_size, stratify=all_y, random_state=seed)
X_train_count = X_count.loc[train_samples]
X_test_count = X_count.loc[test_samples]
y_train = y.loc[train_samples]
y_test = y.loc[test_samples]
print("Train:", X_train_count.shape, "/ Test:", X_test_count.shape)
display(pd.DataFrame({"Train": y_train.value_counts(), "Test": y_test.value_counts()}))
split_df = pd.DataFrame({"sample_name": X_count.index, "split": "Test"})
split_df.loc[split_df["sample_name"].isin(X_train_count.index), "split"] = "Train"
split_df.to_csv(output_dir / "split_assignment.csv", index=False, encoding="utf-8")

### 상대풍부도 Scaler

In [ ]:
scaler = Normalizer(norm="l1")
scaler.fit(X_train_count)
X_train_all = pd.DataFrame(
    scaler.transform(X_train_count),
    index=X_train_count.index, columns=X_train_count.columns)

display(X_train_count.iloc[:3, :4])
display(X_train_all.iloc[:3, :4])

### 학명별 모델 입력 확인

In [ ]:
# unmatched_reads는 정규화에만 사용하고 예측 특성에서는 제외합니다.
X_train = X_train_all[scientific_name_columns]
print("정규화 입력 열:", X_train_all.shape[1], "/ 학명별 모델 특성:", X_train.shape[1])
display(X_train.iloc[:3, :4])

### CV 학습 fold용 특성 선택기

In [ ]:
class FeatureSelector(TransformerMixin, BaseEstimator):
    def __init__(self, feature_columns, use_importance=False, use_vif=False,
                 use_correlation=False, correlation_threshold=0.95,
                 quartile_path=("Q1",), vif_threshold=5.0, n_estimators=300, random_state=42,
                 extra_columns=()):
        self.feature_columns = feature_columns
        self.use_importance = use_importance
        self.use_vif = use_vif
        self.use_correlation = use_correlation
        self.correlation_threshold = correlation_threshold
        self.quartile_path = quartile_path
        self.vif_threshold = vif_threshold
        self.n_estimators = n_estimators
        self.random_state = random_state
        self.extra_columns = extra_columns

    def fit(self, X, y):
        self.feature_names_in_ = X.columns.to_numpy()
        self.n_features_in_ = X.shape[1]
        self.fit_sample_names_ = X.index.tolist()
        columns = list(self.feature_columns)
        self.input_features_ = columns.copy()
        target = (pd.Series(y, index=X.index) == "CRC").astype(int)
        variable = X[columns].columns[X[columns].nunique() > 1]
        correlation = X[variable].corrwith(target).reindex(columns)
        table = correlation.to_frame("target_correlation")
        table["abs_target_correlation"] = correlation.abs()
        table.index.name = "scientific_name"
        self.target_correlation_ = correlation
        self.importance_values_ = pd.Series(dtype=float, name="importance")
        self.importance_cutoff_ = None
        self.resolved_path_ = ()

        # 상수 열과 Spearman 상관이 높은 중복 특성을 제거합니다.
        self.constant_features_ = []
        removed_pairs = []
        if self.use_correlation:
            self.constant_features_ = X[columns].columns[X[columns].nunique() <= 1].tolist()
            columns = X[columns].columns[X[columns].nunique() > 1].tolist()
            pair_correlation = X[columns].corr(method="spearman").abs()
            np.fill_diagonal(pair_correlation.values, 0.0)
            while len(columns) > 1:
                remaining = pair_correlation.loc[columns, columns]
                if remaining.to_numpy().max() <= self.correlation_threshold:
                    break
                i, j = np.unravel_index(remaining.to_numpy().argmax(), remaining.shape)
                first, second = columns[i], columns[j]
                mean_correlation = remaining.sum() / (len(columns) - 1)
                if mean_correlation[first] > mean_correlation[second]:
                    removed = first
                elif mean_correlation[first] < mean_correlation[second]:
                    removed = second
                else:
                    removed = max(first, second)
                removed_pairs.append({
                    "first_feature": first, "second_feature": second,
                    "abs_spearman": remaining.loc[first, second],
                    "first_mean_abs_correlation": mean_correlation[first],
                    "second_mean_abs_correlation": mean_correlation[second],
                    "removed_feature": removed
                })
                columns.remove(removed)
        self.correlation_selected_features_ = columns.copy()
        self.correlation_removed_pairs_ = pd.DataFrame(removed_pairs, columns=[
            "first_feature", "second_feature", "abs_spearman",
            "first_mean_abs_correlation", "second_mean_abs_correlation", "removed_feature"])

        # 타깃 상관이 큰 특성부터 전체 유지 구성의 VIF를 확인합니다.
        self.vif_stage_table_ = pd.DataFrame(columns=["target_correlation", "abs_target_correlation", "VIF"])
        if self.use_vif:
            variable_columns = X[columns].columns[X[columns].nunique() > 1]
            priority = table.loc[variable_columns].rename_axis("scientific_name").sort_values(
                ["abs_target_correlation", "scientific_name"], ascending=[False, True])
            kept = []
            for feature in priority.index:
                trial = kept + [feature]
                if len(trial) >= len(X):
                    break
                vif_input = add_constant(X[trial], has_constant="add").to_numpy()
                if np.linalg.matrix_rank(vif_input) < vif_input.shape[1]:
                    continue
                vif_values = []
                for column in range(1, vif_input.shape[1]):
                    value = variance_inflation_factor(vif_input, column)
                    vif_values.append(value)
                    if not np.isfinite(value) or value >= self.vif_threshold:
                        break
                if np.isfinite(vif_values).all() and max(vif_values) < self.vif_threshold:
                    kept.append(feature)
            columns = kept
            vif_input = add_constant(X[columns], has_constant="add").to_numpy()
            self.vif_stage_table_ = priority.loc[columns, ["target_correlation", "abs_target_correlation"]].copy()
            self.vif_stage_table_["VIF"] = [variance_inflation_factor(vif_input, i) for i in range(1, vif_input.shape[1])]
        self.vif_selected_features_ = columns.copy()

        self.importance_input_features_ = columns.copy()
        # 중요도와 수치 임계값도 이 학습 fold 안에서 계산합니다.
        if self.use_importance:
            importance_model = RandomForestClassifier(
                n_estimators=self.n_estimators, bootstrap=True, oob_score=True,
                class_weight="balanced", random_state=self.random_state, n_jobs=1)
            importance_model.fit(X[columns], y)
            self.importance_values_ = pd.Series(
                importance_model.feature_importances_, index=columns, name="importance")
            values = self.importance_values_[self.importance_values_ > 0]
            for quartile in self.quartile_path:
                if values.nunique() < 2:
                    if self.importance_cutoff_ is None:
                        raise ValueError("양수 중요도를 사분위로 나눌 수 없습니다.")
                    break
                thresholds = values.quantile([0.25, 0.50, 0.75]).to_numpy()
                position = ["Q1", "Q2", "Q3"].index(quartile)
                self.importance_cutoff_ = float(thresholds[position])
                self.resolved_path_ = self.resolved_path_ + (quartile,)
                if position == 0:
                    values = values[values <= thresholds[position]]
                else:
                    values = values[(values > thresholds[position - 1]) & (values <= thresholds[position])]
            columns = self.importance_values_[self.importance_values_ > self.importance_cutoff_].index.tolist()
        self.importance_selected_features_ = columns.copy()

        if not columns:
            raise ValueError("선택된 특성이 없습니다.")
        # 최종 남은 특성의 VIF를 표에 다시 계산합니다.
        self.vif_table_ = self.vif_stage_table_.copy()
        if self.use_vif:
            self.vif_table_ = self.vif_stage_table_.loc[columns].copy()
            if self.use_importance:
                vif_input = add_constant(X[columns], has_constant="add").to_numpy()
                self.vif_table_["VIF"] = [
                    variance_inflation_factor(vif_input, i) for i in range(1, vif_input.shape[1])]
        self.selected_microbiome_features_ = columns.copy()
        self.selected_features_ = columns + list(self.extra_columns)
        return self

    def transform(self, X):
        check_is_fitted(self, "selected_features_")
        return X[self.selected_features_]

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, "selected_features_")
        return np.asarray(self.selected_features_, dtype=object)

### Train 학명별 그룹 분포

In [ ]:
representative_features = X_train.mean().sort_values(ascending=False).head(4).index
plot_df = X_train.copy()
plot_df["DiseaseGroup"] = y_train
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, feature in zip(axes.flat, representative_features):
    sns.boxplot(data=plot_df, x="DiseaseGroup", y=feature, order=class_names, showfliers=False, ax=ax)
    sns.stripplot(data=plot_df, x="DiseaseGroup", y=feature, order=class_names,
                  color="black", alpha=0.5, jitter=False, ax=ax)
    ax.set(title=fill(feature, 35), xlabel="", ylabel="상대풍부도")
    ax.set_ylim(bottom=0)
    ax.tick_params(axis="x", rotation=15)
fig.tight_layout()
fig.savefig(figure_dir / "02_train_feature_distribution.png", dpi=150)
plt.show()

### Train 학명별 상관관계

In [ ]:
correlation_features = X_train.mean().sort_values(ascending=False).head(6).index
correlation_df = X_train[correlation_features].corr(method="spearman")
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_df, annot=True, fmt=".2f", cmap="vlag", vmin=-1, vmax=1)
plt.title("Train의 대표 6개 학명 상관")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(figure_dir / "03_train_correlation.png", dpi=150)
plt.show()

### GridSearchCV·RF·클래스 가중치 설정

In [ ]:
class_counts = y_train.value_counts()
weight_df = pd.DataFrame({
    "Train 샘플 수": class_counts,
    "가중치": len(y_train) / (len(class_counts) * class_counts)
})
display(weight_df.round(3))

# Precision·Recall·F1은 CRC를 양성으로 계산합니다.
scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, pos_label=positive_class, zero_division=0),
    "recall": make_scorer(recall_score, pos_label=positive_class, zero_division=0),
    "f1": make_scorer(f1_score, pos_label=positive_class, zero_division=0)
}
param_grid = {
    "rf__max_depth": [None, 5, 10],
    "rf__min_samples_leaf": [1, 2],
    "rf__max_features": ["sqrt", "log2"]
}
vif_threshold = 5.0
correlation_threshold = 0.95
# 선택용·예측용 RF에 클래스 가중치를 적용합니다.
importance_rf = RandomForestClassifier(
    n_estimators=n_trees, bootstrap=True, oob_score=True,
    class_weight="balanced", random_state=seed, n_jobs=1)
rf = RandomForestClassifier(
    n_estimators=n_trees, bootstrap=True, oob_score=True,
    class_weight="balanced", random_state=seed, n_jobs=1)

# 같은 학습 fold와 선택 규칙은 캐시로 재사용합니다.
cache_memory = joblib.Memory(output_dir / "pipeline_cache", verbose=0)

### Train 전체 RF 중요도 참고

In [ ]:
importance_rf.fit(X_train, y_train)
importance_df = pd.Series(
    importance_rf.feature_importances_,
    index=scientific_name_columns, name="importance").sort_values(ascending=False).to_frame()
importance_df.index.name = "scientific_name"
display(importance_df.head(20))
importance_df.head(20).iloc[::-1].plot.barh(figsize=(10, 8), legend=False)
plt.yticks(fontsize=8)
plt.title("학명별 RF의 Train 중요도 상위 20개")
plt.tight_layout()
plt.savefig(figure_dir / "04_train_importance.png", dpi=150)
plt.show()
importance_df.to_csv(output_dir / "train_feature_importance.csv", encoding="utf-8")

### 1번 · Fold 수·사분위 탐색 설정

In [ ]:
positive_importance = importance_df.loc[importance_df["importance"] > 0, "importance"]
cv_options = [3, 5, 7]
max_refinement_rounds = 6
fold_grids = {}
fold_results = []
cv_tables = []
round_results = []

### 1번 · Fold별 사분위 반복 탐색·학습

In [ ]:
pipeline_importance = Pipeline([
    ("scaler", Normalizer(norm="l1").set_output(transform="pandas")),
    ("selection", FeatureSelector(feature_columns=scientific_name_columns, use_importance=True,
                                  n_estimators=n_trees, random_state=seed)),
    ("rf", rf)
], memory=cache_memory)

for cv_splits in cv_options:
    search_prefix = ()
    best_score = -np.inf
    best_grid = None
    stop_reason = "최대 반복 횟수 도달"
    for search_round in range(1, max_refinement_rounds + 1):
        importance_param_grid = []
        for quartile in ["Q3", "Q2", "Q1"]:
            params = param_grid.copy()
            params["selection__quartile_path"] = [search_prefix + (quartile,)]
            importance_param_grid.append(params)

        grid_search_1 = GridSearchCV(
            estimator=pipeline_importance, param_grid=importance_param_grid,
            cv=cv_splits, scoring=scoring, refit="f1")
        grid_search_1.fit(X_train_count, y_train)

        round_path = grid_search_1.best_params_["selection__quartile_path"]
        round_score = float(grid_search_1.best_score_)
        improved = round_score > best_score + 1e-12
        if improved:
            best_score = round_score
            best_grid = grid_search_1
            best_round = search_round
            best_path = round_path

        round_cv_df = pd.DataFrame(grid_search_1.cv_results_)
        round_cv_df["search_round"] = search_round
        round_cv_df["cv_splits"] = cv_splits
        cv_tables.append(round_cv_df)
        selection = grid_search_1.best_estimator_.named_steps["selection"]
        round_results.append({
            "cv_splits": cv_splits, "stop_reason": "",
            "search_round": search_round, "round_winner": "/".join(round_path),
            "importance_cutoff_train": selection.importance_cutoff_,
            "n_train_features": len(selection.selected_features_),
            "round_cv_f1": round_score, "best_cv_f1": best_score, "improved": improved
        })
        print("Fold:", cv_splits, "/ 회차:", search_round, "/ 선택:", "/".join(round_path), "/ CV F1:", round_score)
        if not improved:
            stop_reason = "CV F1 개선 없음"
            break
        search_prefix = round_path

    round_results[-1]["stop_reason"] = stop_reason
    fold_grids[cv_splits] = best_grid
    selection = best_grid.best_estimator_.named_steps["selection"]
    row = {
        "cv_splits": cv_splits, "n_train_features": len(selection.selected_features_),
        "importance_quartile": "/".join(best_path), "importance_cutoff_train": selection.importance_cutoff_,
        "best_search_round": best_round, "completed_rounds": search_round, "stop_reason": stop_reason,
        "best_params": json.dumps(best_grid.best_params_, ensure_ascii=False)
    }
    for metric in scoring:
        row["cv_" + metric] = best_grid.cv_results_["mean_test_" + metric][best_grid.best_index_]
        row["cv_std_" + metric] = best_grid.cv_results_["std_test_" + metric][best_grid.best_index_]
    fold_results.append(row)
    print("Fold 탐색 완료:", cv_splits, "/ 최고 CV F1:", best_grid.best_score_)

### 1번 · Fold 수 비교·선택

In [ ]:
fold_comparison_df = pd.DataFrame(fold_results)
# F1 동점이면 적은 특성 수, 작은 fold 수를 우선합니다.
best_fold = fold_comparison_df.sort_values(
    ["cv_f1", "n_train_features", "cv_splits"], ascending=[False, True, True], kind="stable").iloc[0]
selected_cv = int(best_fold["cv_splits"])
fold_comparison_df["selected"] = fold_comparison_df["cv_splits"] == selected_cv
best_grid = fold_grids[selected_cv]
best_path = best_grid.best_params_["selection__quartile_path"]
best_round = int(best_fold["best_search_round"])
stop_reason = best_fold["stop_reason"]
display(fold_comparison_df[["cv_splits", "n_train_features", "importance_quartile",
                            "cv_accuracy", "cv_precision", "cv_recall", "cv_f1", "selected"]].round(3))
print("전체 실험에 적용할 fold 수:", selected_cv)
fold_comparison_df.to_csv(output_dir / "cv_fold_comparison.csv", index=False, encoding="utf-8")

### 1번 · Fold 수별 CV 지표 비교

In [ ]:
ax = fold_comparison_df.set_index("cv_splits")[["cv_accuracy", "cv_precision", "cv_recall", "cv_f1"]].plot.bar(
    figsize=(9, 4), rot=0, ylim=(0, 1.12), xlabel="Fold 수", ylabel="CV 점수", width=0.8)
for bars in ax.containers:
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
ax.legend(["Accuracy", "Precision", "Recall", "F1-score"], ncol=4, fontsize=8)
ax.set_title("1번 · Fold 수별 내부 CV 비교")
plt.tight_layout()
plt.savefig(figure_dir / "08_cv_fold_comparison.png", dpi=150)
plt.show()

### 1번 · 최적 모델 확인

In [ ]:
grid_search_1 = best_grid
best_model_1 = grid_search_1.best_estimator_
selected_quartile = "/".join(best_path)
selected_features = best_model_1.named_steps["selection"].selected_features_
importance_cutoff = best_model_1.named_steps["selection"].importance_cutoff_
print("1번 최적 하이퍼파라미터:", grid_search_1.best_params_)
print("선택 경로:", selected_quartile, "/ 학명 수:", len(selected_features))
print("선택된 fold 수:", selected_cv)

### 1번 · Train 기준 임계값 참고표

In [ ]:
threshold_rows = []
for round_table in cv_tables:
    cv_splits = int(round_table["cv_splits"].iloc[0])
    search_round = int(round_table["search_round"].iloc[0])
    candidate_paths = round_table["param_selection__quartile_path"].drop_duplicates().iloc[::-1].tolist()
    # 표시용 Train 임계값입니다. GridSearchCV에는 이 숫자를 전달하지 않습니다.
    for path in candidate_paths:
        values = positive_importance.copy()
        cutoff = None
        resolved = ()
        for quartile in path:
            if values.nunique() < 2:
                break
            thresholds = values.quantile([0.25, 0.50, 0.75]).to_numpy()
            position = ["Q1", "Q2", "Q3"].index(quartile)
            cutoff = float(thresholds[position])
            resolved = resolved + (quartile,)
            if position == 0:
                values = values[values <= thresholds[position]]
            else:
                values = values[(values > thresholds[position - 1]) & (values <= thresholds[position])]
        threshold_rows.append({
            "cv_splits": cv_splits, "search_round": search_round, "quartile": path[-1], "search_path": "/".join(path),
            "resolved_path_train": "/".join(resolved), "importance_cutoff_train": cutoff,
            "n_train_features": int((importance_df["importance"] > cutoff).sum())
        })


threshold_history_df = pd.DataFrame(threshold_rows)

### 1번 · 탐색 기록 저장

In [ ]:
importance_cv_df = pd.concat(cv_tables, ignore_index=True)
round_results_df = pd.DataFrame(round_results)
importance_cv_df.to_csv(output_dir / "gridsearch_1.csv", index=False, encoding="utf-8")
for cv_splits in cv_options:
    fold_tables = [table for table in cv_tables if table["cv_splits"].iloc[0] == cv_splits]
    pd.concat(fold_tables, ignore_index=True).to_csv(
        output_dir / ("gridsearch_1_cv" + str(cv_splits) + ".csv"), index=False, encoding="utf-8")
threshold_history_df.to_csv(output_dir / "importance_quartile_thresholds.csv", index=False, encoding="utf-8")
round_results_df.to_csv(output_dir / "importance_search_rounds.csv", index=False, encoding="utf-8")
print("선택된 fold 탐색 종료:", stop_reason)

### 1번 · 반복 탐색 결과 비교

In [ ]:
quartile_results = []
for _, quartile in threshold_history_df.iterrows():
    path = tuple(quartile["search_path"].split("/"))
    path_match = np.array([p == path for p in importance_cv_df["param_selection__quartile_path"]])
    candidates = importance_cv_df[
        (importance_cv_df["cv_splits"] == quartile["cv_splits"]) &
        (importance_cv_df["search_round"] == quartile["search_round"]) & path_match]
    best_result = candidates.sort_values("rank_test_f1", kind="stable").iloc[0]
    row = quartile.to_dict()
    row["best_params"] = json.dumps(best_result["params"], ensure_ascii=False)
    row["selected"] = (quartile["cv_splits"] == selected_cv) and (quartile["search_round"] == best_round) and (path == best_path)
    for metric in scoring:
        row["cv_" + metric] = best_result["mean_test_" + metric]
    quartile_results.append(row)
quartile_results_df = pd.DataFrame(quartile_results)
display(quartile_results_df[["cv_splits", "search_round", "search_path", "n_train_features", "cv_accuracy",
                             "cv_precision", "cv_recall", "cv_f1", "selected"]].round(3))
display(round_results_df.round(6))
quartile_results_df.to_csv(output_dir / "importance_quartile_comparison.csv", index=False, encoding="utf-8")
quartile_results_df["trial"] = "R" + quartile_results_df["search_round"].astype(str) + "-" + quartile_results_df["quartile"]
quartile_results_df[quartile_results_df["cv_splits"] == selected_cv].set_index("trial")[["cv_accuracy", "cv_precision", "cv_recall", "cv_f1"]].plot(
    figsize=(11, 4), marker="o", ylim=(0, 1.08), ylabel="CV 점수", xlabel="회차·사분위")
plt.title("1번 · 선택된 " + str(selected_cv) + "-fold 사분위 탐색")
plt.xticks(rotation=45)
plt.legend(["Accuracy", "Precision", "Recall", "F1-score"], fontsize=8)
plt.tight_layout()
plt.savefig(figure_dir / "07_importance_quartile_comparison.png", dpi=150)
plt.show()

### Train 학명별 특성·타깃 상관계수

In [ ]:
target_binary = (y_train == positive_class).astype(int)
variable_features = X_train.columns[X_train.nunique() > 1]
target_corr_df = X_train[variable_features].corrwith(target_binary).to_frame("target_correlation")
target_corr_df["abs_target_correlation"] = target_corr_df["target_correlation"].abs()
target_corr_df = target_corr_df.sort_values(
    "abs_target_correlation", ascending=False, kind="stable")
target_corr_df.index.name = "scientific_name"
print("Train에서 값이 변하지 않는 특성:", X_train.shape[1] - len(variable_features))
display(target_corr_df.head(10))
target_corr_df.to_csv(output_dir / "train_feature_target_correlation.csv", encoding="utf-8")

### 2번 · VIF 선택 파이프라인

In [ ]:
pipeline_vif = Pipeline([
    ("scaler", Normalizer(norm="l1").set_output(transform="pandas")),
    ("selection", FeatureSelector(feature_columns=scientific_name_columns, use_vif=True,
                              vif_threshold=vif_threshold)),
    ("rf", rf)
], memory=cache_memory)

### 2번 · GridSearchCV 학습

In [ ]:
grid_search_2 = GridSearchCV(
    estimator=pipeline_vif, param_grid=param_grid,
    cv=selected_cv, scoring=scoring, refit="f1")
grid_search_2.fit(X_train_count, y_train)
print("2번 최적 하이퍼파라미터:", grid_search_2.best_params_)
best_model_2 = grid_search_2.best_estimator_

### 2번 · 선택 특성·탐색 결과 저장

In [ ]:
selection_2 = best_model_2.named_steps["selection"]
vif_2_features = selection_2.selected_features_
vif_2_df = selection_2.vif_table_
print("2번 CV F1:", grid_search_2.best_score_, "/ Train 특성:", len(vif_2_features))
display(vif_2_df.round(6))
vif_2_df.to_csv(output_dir / "vif_2.csv", encoding="utf-8")
pd.DataFrame(grid_search_2.cv_results_).to_csv(output_dir / "gridsearch_2.csv", index=False, encoding="utf-8")

### 3번 · 상수·상관 중복 제거 파이프라인

In [ ]:
pipeline_correlation = Pipeline([
    ("scaler", Normalizer(norm="l1").set_output(transform="pandas")),
    ("selection", FeatureSelector(feature_columns=scientific_name_columns,
                              use_correlation=True, correlation_threshold=correlation_threshold)),
    ("rf", rf)
], memory=cache_memory)

### 3번 · GridSearchCV 학습

In [ ]:
grid_search_3 = GridSearchCV(
    estimator=pipeline_correlation, param_grid=param_grid,
    cv=selected_cv, scoring=scoring, refit="f1")
grid_search_3.fit(X_train_count, y_train)
print("3번 최적 하이퍼파라미터:", grid_search_3.best_params_)
best_model_3 = grid_search_3.best_estimator_

### 3번 · 중복 제거·탐색 결과 저장

In [ ]:
selection_3 = best_model_3.named_steps["selection"]
selected_features_3 = selection_3.selected_features_
correlation_removed_df = selection_3.correlation_removed_pairs_
correlation_3_df = X_train.nunique().to_frame("n_unique_train")
correlation_3_df.index.name = "scientific_name"
correlation_3_df["constant"] = correlation_3_df.index.isin(selection_3.constant_features_)
correlation_3_df["selected"] = correlation_3_df.index.isin(selected_features_3)
print("3번 CV F1:", grid_search_3.best_score_)
print("3번 Train 특성:", len(selection_3.input_features_), "→", len(selected_features_3))
print("상수 제거:", len(selection_3.constant_features_), "/ 상관 중복 제거:", len(correlation_removed_df))
display(correlation_removed_df.head(10).round(6))
correlation_3_df.to_csv(output_dir / "correlation_3.csv", encoding="utf-8")
correlation_removed_df.to_csv(output_dir / "correlation_3_removed_pairs.csv", index=False, encoding="utf-8")
pd.DataFrame(grid_search_3.cv_results_).to_csv(output_dir / "gridsearch_3.csv", index=False, encoding="utf-8")

### 4번 · 상관 중복 제거·중요도 선택 파이프라인

In [ ]:
pipeline_correlation_importance = Pipeline([
    ("scaler", Normalizer(norm="l1").set_output(transform="pandas")),
    ("selection", FeatureSelector(feature_columns=scientific_name_columns,
                              use_correlation=True, correlation_threshold=correlation_threshold,
                              use_importance=True, quartile_path=best_path,
                              n_estimators=n_trees, random_state=seed)),
    ("rf", rf)
], memory=cache_memory)

### 4번 · GridSearchCV 학습

In [ ]:
grid_search_4 = GridSearchCV(
    estimator=pipeline_correlation_importance, param_grid=param_grid,
    cv=selected_cv, scoring=scoring, refit="f1")
grid_search_4.fit(X_train_count, y_train)
print("4번 최적 하이퍼파라미터:", grid_search_4.best_params_)
best_model_4 = grid_search_4.best_estimator_

### 4번 · 선택 특성·탐색 결과 저장

In [ ]:
selection_4 = best_model_4.named_steps["selection"]
correlation_4_features = selection_4.correlation_selected_features_
selected_features_4 = selection_4.selected_features_
correlation_4_df = X_train.nunique().to_frame("n_unique_train")
correlation_4_df.index.name = "scientific_name"
correlation_4_df["constant"] = correlation_4_df.index.isin(selection_4.constant_features_)
correlation_4_df["selected_by_correlation"] = correlation_4_df.index.isin(correlation_4_features)
correlation_4_df["selected"] = correlation_4_df.index.isin(selected_features_4)
importance_4_df = selection_4.importance_values_.to_frame()
importance_4_df.index.name = "scientific_name"
importance_4_df["selected"] = importance_4_df.index.isin(selected_features_4)
print("4번 CV F1:", grid_search_4.best_score_)
print("4번 Train 특성:", len(selection_4.input_features_),
      "→ 중복 제거:", len(correlation_4_features), "→ 중요도:", len(selected_features_4))
print("적용 경로:", selected_quartile, "/ Train 임계값:", selection_4.importance_cutoff_)
display(importance_4_df.sort_values("importance", ascending=False).head(10))
correlation_4_df.to_csv(output_dir / "correlation_4.csv", encoding="utf-8")
selection_4.correlation_removed_pairs_.to_csv(
    output_dir / "correlation_4_removed_pairs.csv", index=False, encoding="utf-8")
importance_4_df.to_csv(output_dir / "importance_4_after_correlation.csv", encoding="utf-8")
pd.DataFrame(grid_search_4.cv_results_).to_csv(output_dir / "gridsearch_4.csv", index=False, encoding="utf-8")

### 5번 · Train 나이·성별 입력

In [ ]:
demographic_columns = ["Age", "Sex"]
sex_mapping = {"female": 0, "male": 1}
X_train_demographics = analysis_df.set_index("sample_name").loc[
    X_train_count.index, demographic_columns].copy()
X_train_demographics["Sex"] = X_train_demographics["Sex"].map(sex_mapping)
X_train_count_5 = pd.concat([X_train_count, X_train_demographics], axis=1)
print("나이·성별 결측:")
display(X_train_demographics.isna().sum())
display(X_train_demographics.head())

### 5번 · 미생물 전처리·나이·성별 결합

In [ ]:
# Normalizer는 미생물 count 열에만 적용합니다.
scaler_5 = ColumnTransformer([
    ("microbiome", Normalizer(norm="l1"), normalization_columns),
    ("demographics", "passthrough", demographic_columns)
], verbose_feature_names_out=False).set_output(transform="pandas")
pipeline_demographics = Pipeline([
    ("scaler", scaler_5),
    ("selection", FeatureSelector(feature_columns=scientific_name_columns,
                              use_importance=True, quartile_path=best_path,
                              n_estimators=n_trees, random_state=seed,
                              extra_columns=demographic_columns)),
    ("rf", rf)
], memory=cache_memory)

### 5번 · GridSearchCV 학습

In [ ]:
grid_search_5 = GridSearchCV(
    estimator=pipeline_demographics, param_grid=param_grid,
    cv=selected_cv, scoring=scoring, refit="f1")
grid_search_5.fit(X_train_count_5, y_train)
print("5번 최적 하이퍼파라미터:", grid_search_5.best_params_)
best_model_5 = grid_search_5.best_estimator_

### 5번 · 선택 특성·탐색 결과 저장

In [ ]:
selection_5 = best_model_5.named_steps["selection"]
selected_features_5 = selection_5.selected_features_
selected_features_5_df = pd.DataFrame({"feature_name": selected_features_5})
selected_features_5_df["feature_type"] = "microbiome"
selected_features_5_df.loc[
    selected_features_5_df["feature_name"].isin(demographic_columns), "feature_type"] = "demographic"
print("5번 CV F1:", grid_search_5.best_score_)
print("미생물 특성:", len(selection_5.selected_microbiome_features_),
      "+ 나이·성별:", len(demographic_columns), "= 전체:", len(selected_features_5))
print("적용 경로:", selected_quartile, "/ Train 임계값:", selection_5.importance_cutoff_)
display(selected_features_5_df.tail())
selected_features_5_df.to_csv(output_dir / "selected_features_5.csv", index=False, encoding="utf-8")
X_train_demographics.to_csv(output_dir / "train_demographics.csv", encoding="utf-8")
pd.DataFrame(grid_search_5.cv_results_).to_csv(output_dir / "gridsearch_5.csv", index=False, encoding="utf-8")
joblib.dump(best_model_5.named_steps["scaler"], output_dir / "5_scaler.joblib")

### 주요 학명별 특성 대응 확인

In [ ]:
important_feature_df = pd.merge(
    importance_df.head(10).reset_index(), feature_annotation_df.reset_index(), on="scientific_name")
display(important_feature_df)
important_feature_df.to_csv(output_dir / "important_feature_annotation.csv", index=False, encoding="utf-8")

### CV·OOB 지표 계산·저장

In [ ]:
model_list = [
    ("1", "양수 중요도 " + selected_quartile, grid_search_1),
    ("2", "VIF 5 미만", grid_search_2),
    ("3", "상수·Spearman 상관 중복 제거", grid_search_3),
    ("4", "상관 중복 제거 + 중요도 " + selected_quartile, grid_search_4),
    ("5", "1번 중요도 + 나이·성별", grid_search_5)
]
results = []
prediction_tables = []
feature_tables = []
for experiment_id, change, grid in model_list:
    pipeline = grid.best_estimator_
    selection = pipeline.named_steps["selection"]
    fitted_rf = pipeline.named_steps["rf"]
    selected_columns = selection.selected_features_
    oob_prob = fitted_rf.oob_decision_function_
    oob_pred = fitted_rf.classes_[oob_prob.argmax(axis=1)]
    print(experiment_id, "OOB 확률 없는 샘플:", (oob_prob.sum(axis=1) == 0).sum())
    row = {
        "experiment_id": experiment_id, "change": change, "scope": "CV 학습 fold 내 선택",
        "cv_splits": int(grid.cv), "input_type": "relative_abundance_and_demographics" if experiment_id == "5" else "relative_abundance", "n_features": len(selected_columns),
        "classification_task": "Normal_vs_CRC", "class_weight": "balanced", "positive_class": positive_class,
        "best_params": json.dumps(grid.best_params_, ensure_ascii=False),
        "importance_quartile": selected_quartile if experiment_id in ["1", "4", "5"] else None,
        "importance_cutoff_train": selection.importance_cutoff_,
        "correlation_threshold": correlation_threshold if experiment_id in ["3", "4"] else None,
        "oob_accuracy": accuracy_score(y_train, oob_pred),
        "oob_precision": precision_score(y_train, oob_pred, pos_label=positive_class, zero_division=0),
        "oob_recall": recall_score(y_train, oob_pred, pos_label=positive_class, zero_division=0),
        "oob_f1": f1_score(y_train, oob_pred, pos_label=positive_class, zero_division=0)
    }
    for metric in scoring:
        row["cv_" + metric] = grid.cv_results_["mean_test_" + metric][grid.best_index_]
        row["cv_std_" + metric] = grid.cv_results_["std_test_" + metric][grid.best_index_]
    results.append(row)
    predictions = pd.DataFrame({"sample_name": y_train.index, "actual": y_train.values,
                                "predicted": oob_pred, "experiment_id": experiment_id})
    for i, label in enumerate(fitted_rf.classes_):
        predictions["probability_" + label] = oob_prob[:, i]
    prediction_tables.append(predictions)
    features = pd.DataFrame({"experiment_id": experiment_id, "feature_name": selected_columns})
    features["feature_type"] = "microbiome"
    features.loc[features["feature_name"].isin(demographic_columns), "feature_type"] = "demographic"
    feature_tables.append(features)
results_df = pd.DataFrame(results)
oob_predictions_df = pd.concat(prediction_tables, ignore_index=True)
selected_features_df = pd.concat(feature_tables, ignore_index=True)
display(results_df[["experiment_id", "cv_splits", "n_features", "best_params"]])
display(results_df[["experiment_id", "cv_accuracy", "cv_precision", "cv_recall", "cv_f1"]].round(3))
display(results_df[["experiment_id", "oob_accuracy", "oob_precision", "oob_recall", "oob_f1"]].round(3))
results_df.to_csv(output_dir / "experiment_results.csv", index=False, encoding="utf-8")
oob_predictions_df.to_csv(output_dir / "oob_predictions.csv", index=False, encoding="utf-8")
selected_features_df.to_csv(output_dir / "selected_features.csv", index=False, encoding="utf-8")

### CV·OOB 지표 비교

In [ ]:
# CV와 OOB를 같은 네 지표로 비교합니다.
metrics = ["accuracy", "precision", "recall", "f1"]
titles = ["Accuracy", "Precision", "Recall", "F1-score"]
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, metric, title in zip(axes.flat, metrics, titles):
    results_df.set_index("experiment_id")[["cv_" + metric, "oob_" + metric]].plot.bar(
        ax=ax, rot=0, ylim=(0, 1.12), title=title, xlabel="", ylabel="점수", width=0.8)
    ax.set_yticks(np.linspace(0, 1, 6))
    for bars in ax.containers:
        ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    ax.get_legend().remove()
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, ["CV 평균 (" + str(selected_cv) + "-fold)", "OOB 참고"], loc="upper center",
           bbox_to_anchor=(0.5, 0.955), ncol=2, frameon=False, fontsize=9)
fig.suptitle("CV: fold별 특성 선택 포함 · OOB: 참고용", y=0.995)
fig.tight_layout(rect=(0, 0, 1, 0.91))
fig.savefig(figure_dir / "05_cv_oob_comparison.png", dpi=150)
plt.show()

### OOB 클래스별 점수·혼동행렬

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, experiment_id in zip(axes.flat, results_df["experiment_id"]):
    predictions = oob_predictions_df[oob_predictions_df["experiment_id"] == experiment_id]
    print(experiment_id)
    report_df = pd.DataFrame(classification_report(
        predictions["actual"], predictions["predicted"], labels=class_names,
        zero_division=0, output_dict=True)).T
    report_df = report_df.loc[class_names, ["precision", "recall", "f1-score", "support"]]
    display(report_df.round(3))
    ConfusionMatrixDisplay.from_predictions(
        predictions["actual"], predictions["predicted"], labels=class_names,
        xticks_rotation=20, colorbar=False, ax=ax)
    ax.set_title(experiment_id + " OOB 참고")
    ax.grid(False)
axes.flat[-1].set_visible(False)
fig.tight_layout()
fig.savefig(figure_dir / "06_oob_confusion.png", dpi=150)
plt.show()

### Test 입력

In [ ]:
# 고정된 Test 샘플을 사용합니다.
X_test_demographics = analysis_df.set_index("sample_name").loc[
    X_test_count.index, demographic_columns].copy()
X_test_demographics["Sex"] = X_test_demographics["Sex"].map(sex_mapping)
X_test_count_5 = pd.concat([X_test_count, X_test_demographics], axis=1)
print("Test 샘플 수:", len(y_test))
display(y_test.value_counts())
display(X_test_demographics.isna().sum())

### 1번 · Test 예측

In [ ]:
test_rf_1 = best_model_1.named_steps["rf"]
X_test_1 = best_model_1.named_steps["scaler"].transform(X_test_count)
X_test_1 = X_test_1[test_rf_1.feature_names_in_]
test_pred_1 = test_rf_1.predict(X_test_1)
test_prob_1 = test_rf_1.predict_proba(X_test_1)

### 2번 · Test 예측

In [ ]:
test_rf_2 = best_model_2.named_steps["rf"]
X_test_2 = best_model_2.named_steps["scaler"].transform(X_test_count)
X_test_2 = X_test_2[test_rf_2.feature_names_in_]
test_pred_2 = test_rf_2.predict(X_test_2)
test_prob_2 = test_rf_2.predict_proba(X_test_2)

### 3번 · Test 예측

In [ ]:
test_rf_3 = best_model_3.named_steps["rf"]
X_test_3 = best_model_3.named_steps["scaler"].transform(X_test_count)
X_test_3 = X_test_3[test_rf_3.feature_names_in_]
test_pred_3 = test_rf_3.predict(X_test_3)
test_prob_3 = test_rf_3.predict_proba(X_test_3)

### 4번 · Test 예측

In [ ]:
test_rf_4 = best_model_4.named_steps["rf"]
X_test_4 = best_model_4.named_steps["scaler"].transform(X_test_count)
X_test_4 = X_test_4[test_rf_4.feature_names_in_]
test_pred_4 = test_rf_4.predict(X_test_4)
test_prob_4 = test_rf_4.predict_proba(X_test_4)

### 5번 · Test 예측

In [ ]:
test_rf_5 = best_model_5.named_steps["rf"]
X_test_5 = best_model_5.named_steps["scaler"].transform(X_test_count_5)
X_test_5 = X_test_5[test_rf_5.feature_names_in_]
test_pred_5 = test_rf_5.predict(X_test_5)
test_prob_5 = test_rf_5.predict_proba(X_test_5)

### Test 네 지표·예측 저장

In [ ]:
test_models = [
    ("1", test_rf_1, test_pred_1, test_prob_1),
    ("2", test_rf_2, test_pred_2, test_prob_2),
    ("3", test_rf_3, test_pred_3, test_prob_3),
    ("4", test_rf_4, test_pred_4, test_prob_4),
    ("5", test_rf_5, test_pred_5, test_prob_5)
]
test_results = []
test_prediction_tables = []
for experiment_id, model, predicted, probability in test_models:
    test_results.append({
        "experiment_id": experiment_id, "n_features": model.n_features_in_, "n_test": len(y_test),
        "scope": "test", "positive_class": positive_class,
        "test_accuracy": accuracy_score(y_test, predicted),
        "test_precision": precision_score(y_test, predicted, pos_label=positive_class, zero_division=0),
        "test_recall": recall_score(y_test, predicted, pos_label=positive_class, zero_division=0),
        "test_f1": f1_score(y_test, predicted, pos_label=positive_class, zero_division=0)
    })
    predictions = pd.DataFrame({"experiment_id": experiment_id, "sample_name": y_test.index,
                                "actual": y_test.values, "predicted": predicted})
    for i, label in enumerate(model.classes_):
        predictions["probability_" + label] = probability[:, i]
    test_prediction_tables.append(predictions)
test_results_df = pd.DataFrame(test_results)
test_predictions_df = pd.concat(test_prediction_tables, ignore_index=True)
display(test_results_df[["experiment_id", "n_features", "test_accuracy", "test_precision",
                         "test_recall", "test_f1"]].round(3))
test_results_df.to_csv(output_dir / "test_metrics.csv", index=False, encoding="utf-8")
test_predictions_df.to_csv(output_dir / "test_predictions.csv", index=False, encoding="utf-8")
test_evaluation_info = {
    "test_reevaluated": True, "test_evaluation_requested_by_user": True,
    "test_evaluation_scope": f"fixed Test split of {len(y_test)} patients; model comparison",
    "test_evaluated_experiments": test_results_df["experiment_id"].tolist(),
    "test_sample_count": len(y_test), "test_model_selection_performed": False
}

### CV·OOB·Test 비교

In [ ]:
test_metric_columns = ["test_accuracy", "test_precision", "test_recall", "test_f1"]
evaluation_comparison_df = pd.merge(
    results_df, test_results_df[["experiment_id"] + test_metric_columns], on="experiment_id", how="left")
evaluation_comparison_df.to_csv(output_dir / "evaluation_comparison.csv", index=False, encoding="utf-8")
metrics = ["accuracy", "precision", "recall", "f1"]
titles = ["Accuracy", "Precision", "Recall", "F1-score"]
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, metric, title in zip(axes.flat, metrics, titles):
    evaluation_comparison_df.set_index("experiment_id")[["cv_" + metric, "oob_" + metric, "test_" + metric]].plot.bar(
        ax=ax, rot=0, ylim=(0, 1.12), title=title, xlabel="", ylabel="점수", width=0.8)
    ax.set_yticks(np.linspace(0, 1, 6))
    for bars in ax.containers:
        ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=7)
    ax.get_legend().remove()
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, ["CV 평균 (" + str(selected_cv) + "-fold)", "OOB 참고", f"Test {len(y_test)}명"],
           loc="upper center", bbox_to_anchor=(0.5, 0.955), ncol=3, frameon=False, fontsize=9)
fig.suptitle("1~5번 · CV·OOB·Test 비교", y=0.995)
fig.tight_layout(rect=(0, 0, 1, 0.91))
fig.savefig(figure_dir / "09_cv_oob_test_comparison.png", dpi=150)
plt.show()

### Test 클래스별 점수·혼동행렬

In [ ]:
test_report_tables = []
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, (experiment_id, model, predicted, probability) in zip(axes.flat, test_models):
    report_df = pd.DataFrame(classification_report(
        y_test, predicted, labels=class_names, zero_division=0, output_dict=True)).T
    report_df = report_df.loc[class_names, ["precision", "recall", "f1-score", "support"]]
    report_df.index.name = "class"
    report_df = report_df.reset_index()
    report_df.insert(0, "experiment_id", experiment_id)
    test_report_tables.append(report_df)
    ConfusionMatrixDisplay.from_predictions(
        y_test, predicted, labels=class_names, xticks_rotation=20, colorbar=False, ax=ax)
    ax.set_title(experiment_id + " Test")
    ax.grid(False)
test_class_reports_df = pd.concat(test_report_tables, ignore_index=True)
display(test_class_reports_df.round(3))
test_class_reports_df.to_csv(output_dir / "test_class_reports.csv", index=False, encoding="utf-8")
axes.flat[-1].set_visible(False)
fig.tight_layout()
fig.savefig(figure_dir / "10_test_confusion.png", dpi=150)
plt.show()

### Streamlit 모델 선택

In [ ]:
selected_id = "1"
print("Streamlit 모델: 7:3 분할의 실험 1번")

### Scaler·모델·실행 설정 저장

In [ ]:
# 추론용 모델에는 학습된 RF와 선택 학명만 저장합니다.
for experiment_id, change, grid in model_list:
    pipeline = grid.best_estimator_
    selection = pipeline.named_steps["selection"]
    selected_columns = selection.selected_features_
    model = Pipeline([("rf", pipeline.named_steps["rf"])])
    extra_columns = demographic_columns if experiment_id == "5" else []
    selected_annotation = feature_annotation_df.reindex(selected_columns).rename_axis("feature_name").reset_index()
    selected_annotation["feature_type"] = "microbiome"
    selected_annotation.loc[selected_annotation["feature_name"].isin(extra_columns), "feature_type"] = "demographic"
    model_bundle = {
        "model": model, "scaler": pipeline.named_steps["scaler"],
        "scaler_file": "5_scaler.joblib" if experiment_id == "5" else "final_scaler.joblib",
        "scaler_input_type": "scientific_name_read_count_and_demographics" if experiment_id == "5" else "scientific_name_read_count",
        "scaler_input_columns": normalization_columns + extra_columns,
        "normalization_columns": normalization_columns, "demographic_columns": extra_columns,
        "sex_mapping": sex_mapping if experiment_id == "5" else None, "age_transform": "passthrough" if experiment_id == "5" else None,
        "normalization_only_columns": ["unmatched_reads"],
        "feature_columns": scientific_name_columns + extra_columns,
        "microbiome_feature_columns": scientific_name_columns,
        "selected_microbiome_feature_columns": selection.selected_microbiome_features_, "selected_feature_columns": selected_columns,
        "model_input_columns": selected_columns, "input_type": "scientific_name_relative_abundance_and_demographics" if experiment_id == "5" else "scientific_name_relative_abundance",
        "model_classes": model.classes_.tolist(), "classification_task": "Normal_vs_CRC",
        "class_weight": "balanced", "positive_class": positive_class, "target_mapping": target_mapping,
        "blast_hit_rule": "highest bit_score; first row for ties",
        "sequence_scientific_name_mapping": asv_annotation_df[["ASV_ID", "sequence", "subject_scientific_name"]],
        "asv_sequence_mapping": taxonomy_df[["ASV_ID", "sequence"]],
        "feature_aggregation_rule": "sum all matched ASV reads by subject_scientific_name",
        "aggregation_uses_target": False,
        "relative_abundance_denominator": "전체 원본 ASV count 합 (학명별 합계 + 미매칭 read)",
        "training_sample_names": y_train.index.tolist(), "training_sample_count": len(y_train),
        "train_test_test_size": test_size, "feature_selection": change,
        "selection_order": ["importance", "append_demographics"] if experiment_id == "5" else
                           ["constant", "correlation", "importance"] if experiment_id == "4" else
                           ["constant", "correlation"] if experiment_id == "3" else
                           ["vif"] if experiment_id == "2" else ["importance"],
        "vif_selected_features": selection.vif_selected_features_ if experiment_id == "2" else None,
        "importance_input_features": selection.importance_input_features_ if experiment_id in ["1", "4", "5"] else None,
        "correlation_selected_features": selection.correlation_selected_features_ if experiment_id in ["3", "4"] else None,
        "selected_feature_annotation": selected_annotation,
        "selection_scope": "CV 학습 fold 내 선택", "cv_scope": "internal tuning with foldwise feature selection",
        "oob_scope": "reference after full-Train feature selection",
        "vif_threshold": vif_threshold if experiment_id == "2" else None,
        "correlation_method": "spearman" if experiment_id in ["3", "4"] else None,
        "correlation_threshold": correlation_threshold if experiment_id in ["3", "4"] else None,
        "correlation_removal_rule": "higher mean absolute correlation; alphabetical last on ties" if experiment_id in ["3", "4"] else None,
        "constant_features": selection.constant_features_ if experiment_id in ["3", "4"] else None,
        "importance_cutoff_train": selection.importance_cutoff_,
        "importance_quartile": selected_quartile if experiment_id in ["1", "4", "5"] else None,
        "best_params": grid.best_params_, "cv_splits": int(grid.cv), "refit_metric": "f1", "experiment_id": experiment_id
    }
    joblib.dump(model_bundle, output_dir / (experiment_id + "_model.joblib"))
joblib.dump(grid_search_1.best_estimator_.named_steps["scaler"], output_dir / "final_scaler.joblib")
run_info = {
    "seed": seed, "test_size": test_size, "train_test_ratio": "7:3", "stratify_column": "DiseaseStatus",
    "n_train": len(y_train), "n_test": len(y_test), "train_samples": y_train.index.tolist(), "test_samples": y_test.index.tolist(),
    "experiments": results_df["experiment_id"].tolist(), "cv_splits": selected_cv,
    "experiment1_cv_options": cv_options, "experiment1_selected_cv": selected_cv,
    "cv_selection_rule": "CV F1 descending, feature count ascending, fold count ascending",
    "classification_task": "Normal_vs_CRC", "class_names": class_names, "class_weight": "balanced",
    "positive_class": positive_class, "target_mapping": target_mapping,
    "n_samples": len(analysis_df), "n_raw_asvs": raw_count_df.shape[1],
    "n_blast_matched_asvs": len(asv_annotation_df), "n_scientific_name_features": len(scientific_name_columns),
    "n_scaler_input_columns": len(normalization_columns), "normalization_only_columns": ["unmatched_reads"],
    "aggregation_uses_target": False, "feature_aggregation_rule": "sum all matched ASV reads by subject_scientific_name",
    "param_grid": param_grid, "refit_metric": "f1", "max_refinement_rounds": max_refinement_rounds,
    "completed_refinement_rounds": int(best_fold["completed_rounds"]), "total_refinement_rounds": len(round_results_df), "selected_importance_search_round": best_round,
    "selected_importance_quartile": selected_quartile, "selected_importance_cutoff_train": importance_cutoff,
    "importance_search_stop_reason": stop_reason,
    "scaler_type": "Normalizer", "scaler_norm": "l1", "scaler_file": "final_scaler.joblib",
    "feature_selection_within_cv": {"1": True, "2": True, "3": True, "4": True, "5": True},
    "importance_cutoff_within_cv": True, "vif_selection_within_cv": {"2": True},
    "correlation_selection_within_cv": {"3": True, "4": True}, "correlation_method": "spearman",
    "correlation_threshold": correlation_threshold, "correlation_selection_uses_target": False,
    "evaluation_phases": ["CV", "OOB", "Test"], "training_metrics_computed": False,
    "cv_scope": "internal tuning with foldwise feature selection", "oob_scope": "reference after full-Train feature selection",
    "vif_threshold": vif_threshold, "selected_experiment": selected_id,
    "model_selection_source": "user", "selected_model_file": selected_id + "_model.joblib",
    "selection_order": {"1": ["importance"], "2": ["vif"], "3": ["constant", "correlation"], "4": ["constant", "correlation", "importance"], "5": ["importance", "append_demographics"]},
    "experiment4_importance_path_source": "experiment1 selected quartile path",
    "experiment4_importance_cutoff_scope": "computed after correlation removal within each training fold",
    "experiment5_demographic_columns": demographic_columns, "experiment5_sex_mapping": sex_mapping,
    "experiment5_age_transform": "passthrough", "experiment5_scaler_file": "5_scaler.joblib",
    "experiment5_demographics_in_normalization": False, "experiment5_demographics_in_feature_selection": False,
    "streamlit_model_selection": selected_id, "test_reevaluated": True, "exported_model_requires_custom_selector": False
}
run_info.update(test_evaluation_info)
(output_dir / "run_manifest.json").write_text(
    json.dumps(run_info, ensure_ascii=False, indent=2), encoding="utf-8")